# **Image Stitching con Keypoint SIFT**
Obiettivo dell'esercitazione è la realizzazione di un metodo di image stitching basato sull'uso di keypoint e descrittori locali.

In particolare, date due immagini in input che rappresentano la stessa scena da punti di vista diversi, l'algoritmo dovrà creare un'immagine panoramica come composizione delle due immagini parziali.

Per l'estazione di keypoint e descrittori locali si utilizzerà la tecnica **SIFT (Scale-Invariant Feature Transform)**, mentre per il matching useremo l'algoritmo **Ransac**.

# **Import delle librerie**
È necessario eseguire l'import delle librerie utilizzate durante l'esercitazione.

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from skimage import exposure
import matplotlib.pyplot as plt

# **Caricamento e visualizzazione immagini**

In [ ]:
img1 = cv2.imread('s1.jpg')
img2 = cv2.imread('s2.jpg')
cv2_imshow(cv2.hconcat([img1, img2]))

# **Detection dei keypoint SIFT e calcolo dei descrittori locali**
I keypoint e relativi descrittori vengono calcolati sulle immagini **grayscale**, quindi è necessaria una conversione delle due immagini in input.

L'estrazione dei keypoint e il calcolo dei descrittori avviene tramite l'oggetto SIFT della libreria cv2.
La classe SIFT deriva dalla classe astratta [**Feature2D**](https://docs.opencv.org/4.12.0/d0/d13/classcv_1_1Feature2D.html#details) che rappresenta il **concetto astratto di un rilevatore e descrittore di caratteristiche locali** in un’immagine.

L'oggetto SIFT può essere istanziato richiamando il metodo
```
cv2.SIFT.create()
```
[Qui](https://docs.opencv.org/4.12.0/d7/d60/classcv_1_1SIFT.html#a4264f700a8133074fb477e30d9beb331) sono disponibili approfondimenti su questo metodo e sui suoi parametri. In questa esercitazione utilizzeremo i **parametri di default**.

Per la detection dei keypoint e dei relativi descrittori in ciascuna immagine utilizzeremo il metodo
```
detectAndCompute()
```
dell'oggetto SIFT istanziato precedentemente. Al metodo dovremo passare l'immagine grayscale e un'eventuale maschera binaria (None nel nostro caso). [Qui](https://docs.opencv.org/4.12.0/d0/d13/classcv_1_1Feature2D.html#a8be0d1c20b08eb867184b8d74c15a677) sono disponibili approfondimenti sul metodo.



In [ ]:
# Conversione in scala di grigi
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# Estrazione dei keypoints e dei descrittori SIFT
sift = cv2.SIFT.create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

# Visualizzazione delle due immagini e dei relativi keypoint
cv2_imshow(cv2.hconcat([cv2.drawKeypoints(gray1, kp1, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS),
                        cv2.drawKeypoints(gray2, kp2, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)]))

# **Matching dei keypoint**
Per il matching dei keypoint è necessario:


1.   Individuare i match presunti usando uno dei [metodi](https://docs.opencv.org/4.12.0/d8/d9b/group__features2d__match.html) messi a disposizione dalla libreria: la ricerca **esaustiva**, esatta ma meno efficiente, è implementata nel Brute Force Matcher (**BFMatcher**) mentre la ricerca **approssimata** tramite indice Kd-Tree, approssimata ma più veloce, è implementata nel Flann Matcher (**FlannBasedMatcher**). Per istanziare il matcher richiamare il metodo` create()`.
Per ogni keypoint, cerchiamo i due descrittori più vicini nell'altra immagine (k=2) richiamando il metodo `knnMatch` del matcher. Al metodo dovranno essere passati i **descrittori delle due immagini e il valore di k**. Il metodo restituisce un array che contiene, per ogni descrittore, i due match migliori e la relativa distanza.

2.   Filtrare i match usando il **Lowe's ratio test** (rapporto tra le distanze dei due Nearest Neighbor); come soglia possiamo utilizzare 0.7, ma è interessante fare qualche prova al variare di questo parametro.



In [ ]:
# Flann Matcher che fa uso di Kd-Tree per una ricerca più efficiente (non esatta)
# matcher = cv2.FlannBasedMatcher.create()

# Ricerca delle corrispondenze tra descrittori usando BFMatcher con L2 norm
matcher = cv2.BFMatcher.create()

# Matching dei descrittori
matches = matcher.knnMatch(des1, des2, k=2)

# Filtraggio dei match sulla base del criterio di Lowe
good_matches = []
for m, n in matches:
    if m.distance/n.distance < 0.7:
        good_matches.append(m)

# Visualizzazione dei match
img_matches = cv2.drawMatches(img1, kp1, img2, kp2, good_matches[:50], None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
cv2_imshow(img_matches)

# **Calcolo omografia, trasformazione geometrica e stitching**



Una volta individuati i good match, possiamo calcolare l'omografia e applicarla alla prima immagine in modo da allinearla alla seconda.
Per il calcolo dell'omografia usiamo la funzione **`cv2.findHomography`**, descritta [qui](https://docs.opencv.org/4.12.0/d9/d0c/group__calib3d.html#ga4abc2ece9fab9398f2e560d53c8c9780). I principali parametri del metodo sono:
*   `srcPoints`: coordinate spaziali dei punti nella prima immagine;
*   `dstPoints`: coordinate spaziali dei punti nella seconda immagine;
*   `method`:	metodo usato per il calcolo dell'omografia (noi useremo il RANSAC)
*   `ransacReprojThreshold`: soglia (in numero di pixel) utilizzata per individuare gli inlier (di solito tra 1 e 10).

Per il calcolo delle coordinate trasformate di un insieme di punti possiamo utilizzare il metodo `cv2.perspectiveTransform` descritta [qui](https://docs.opencv.org/4.12.0/d2/de8/group__core__array.html#gad327659ac03e5fd6894b90025e6900a7).

Il warping (trasformazione geometrica) vero e proprio dell'immagine si effettua richiamando il metodo `cv2.warpPerspective`.




In [ ]:
# Controllo sul numero di corrispondenze trovate
if len(good_matches) > 3:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Calcolo della matrice di omografia utilizzando RANSAC
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 1.0)

    # Calcolo dei bordi dell'immagine trasformata
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    corners_img1 = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)

    # Calcoliamo le coordinate trasformate dei corner dell'immagine
    corners_img1_transformed = cv2.perspectiveTransform(corners_img1, H)
    corners = np.concatenate((corners_img1_transformed, np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)), axis=0)

    # Calcolo dei limiti dell'immagine panoramica
    [xmin, ymin] = np.int32(corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(corners.max(axis=0).ravel() + 0.5)

    # Traslazione necessaria per evitare coordinate negative
    translation = [-xmin, -ymin]
    H_translation = np.array([[1, 0, translation[0]], [0, 1, translation[1]], [0, 0, 1]])

    # Allineamento di img1 e img2 nello spazio della panoramica. Applichiamo all'immagine contemporaneamente la traslazione e l'omografia.
    panorama = cv2.warpPerspective(img1, H_translation.dot(H), (xmax - xmin, ymax - ymin))
    panorama[translation[1]:h2 + translation[1], translation[0]:w2 + translation[0]] = img2
    cv2_imshow(panorama)

# **Lo Stitcher di OpenCV**

La libreria OpenCV mette già a disposizione uno Stitcher, la cui documentazione è disponibile [qui](https://docs.opencv.org/4.12.0/d2/d8d/classcv_1_1Stitcher.html) che include una fase di compensazione della luminosità.

In [ ]:
# Uso dello stitcher di opencv
stitcher = cv2.Stitcher.create()
img3 = cv2.imread('s3.jpg')
(status, stitched) = stitcher.stitch([img1, img2])
if status == 0:
    cv2_imshow(stitched)
else:
    print("Stitching non eseguito")